# 05 · Сравнение методов

Несколько вариантов LoRA на одних данных с общей базовой линией.

Правило: **каждый прогон начинается с чистой загрузки**. Иначе адаптер предыдущего метода протекает в следующий замер, и сравнение недействительно.

Четыре числа на метод, все важны: обучаемые параметры, потери, время, пиковая память. Метод, выигравший по потерям ценой пятикратного времени, на итерациях проиграет.

In [ ]:
from common import MODEL_ID, SYSTEM, DATA, RUNS, policy_suite, tools_suite

import json, torch
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from vlmkit import ChatCollator, cleanup, load_jsonl, evaluate as ev
from vlmkit.compat import supported, first_accepted

TRAIN = load_jsonl(DATA / "policy.jsonl")[::2] + load_jsonl(DATA / "tools.jsonl")[::2]
SUITES = [policy_suite(), tools_suite()]
TARGETS = r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"

BASE_ARGS = dict(
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=2, learning_rate=1e-4, lr_scheduler_type="cosine",
    bf16=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused", logging_steps=10, save_strategy="no",
    remove_unused_columns=False, report_to=[], seed=42,
)
BASE_ARGS |= first_accepted(TrainingArguments, {"warmup_ratio": 0.03, "warmup_steps": 5})

In [ ]:
def fresh():
    m = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
        attn_implementation="sdpa", trust_remote_code=True)
    p = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
    return m, p


def one_run(name, **lora_kwargs):
    """Чистая загрузка → адаптер → обучение → замер → очистка."""
    torch.cuda.reset_peak_memory_stats()
    model, processor = fresh()

    model = get_peft_model(model, LoraConfig(
        target_modules=TARGETS, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", **lora_kwargs))
    model.enable_input_require_grads()
    model.config.use_cache = False

    trainer = Trainer(
        model=model,
        args=TrainingArguments(**supported(TrainingArguments, {**BASE_ARGS, "output_dir": str(RUNS / name)})),
        train_dataset=TRAIN, data_collator=ChatCollator(processor, system=SYSTEM))
    out = trainer.train()

    row = {
        "run": name,
        "обуч.M": round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 1),
        "loss": round(out.training_loss, 3),
        "мин": round(out.metrics.get("train_runtime", 0) / 60, 1),
        "GiB": round(torch.cuda.max_memory_allocated() / 2**30, 1),
    }
    model.eval()
    for s in SUITES:
        m = ev.run(model, processor, s)
        row[s.name] = f"{m['hit']:.0%}/{m['false']:.0%}"

    del trainer, model, processor
    cleanup()
    return row


def baseline():
    model, processor = fresh()
    row = {"run": "базовая", "обуч.M": 0, "loss": None, "мин": 0, "GiB": 0}
    for s in SUITES:
        m = ev.run(model, processor, s)
        row[s.name] = f"{m['hit']:.0%}/{m['false']:.0%}"
    del model, processor
    cleanup()
    return row

## План

Пары подобраны так, чтобы каждая отвечала на один вопрос:

- `lora-r32` против `rslora-r32` — чистый эффект масштаба `α/√r` при одном ранге;
- `rslora-r32` против `rslora-r128` — даёт ли высокий ранг прирост, когда масштаб исправлен;
- `dora-r16` — низкий ранг, где DoRA сильнее всего.

In [ ]:
PLAN = [
    ("lora-r32",    dict(r=32,  lora_alpha=64,  use_rslora=False)),
    ("rslora-r32",  dict(r=32,  lora_alpha=64,  use_rslora=True)),
    ("rslora-r128", dict(r=128, lora_alpha=256, use_rslora=True)),
    ("dora-r16",    dict(r=16,  lora_alpha=32,  use_rslora=True, use_dora=True)),
]

rows = [baseline()]
print(rows[0])
for name, kw in PLAN:
    print(f"\n{'═' * 60}\n{name}")
    rows.append(one_run(name, **kw))
    print(rows[-1])

RUNS.mkdir(exist_ok=True)
(RUNS / "sweep.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
cols = list(rows[0])
print(" ".join(f"{c:>11}" for c in cols))
print("─" * (12 * len(cols)))
for r in rows:
    print(" ".join(f"{str(r[c]):>11}" for c in cols))

## Как читать

В колонках наборов — `попадание/ложные`. Метод со 100% попадания и 50% ложных просто научил модель применять поведение везде.

Если `rslora-r128` не лучше `rslora-r32` — для этой задачи ранга 32 хватает, и это нормально: правила поведения низкоранговые по природе. Разница между `lora-r32` и `rslora-r32` при одном ранге должна быть небольшой — эффект масштаба проявляется на высоких рангах, а не на 32.